# 03 Preference Pair Audit

Purpose: generate task-specific preference pairs from the current SFT adapter, then inspect the emitted DPO dataset and audit trail directly from the Drive-backed runtime outputs.

This notebook assumes `notebooks/00_colab_setup.ipynb` has already mounted Drive and installed the Colab runtime dependencies from `json-ft-source`.


In [1]:
from pathlib import Path
import subprocess

RUN_NAME = "pref-dev-colab"
SOURCE_DATASET = "/content/drive/MyDrive/json-ft-source/data/fixtures/support_tickets.jsonl"
RUNTIME_ROOT = "/content/drive/MyDrive/json-ft-runs"

command = [
    "python",
    "/content/drive/MyDrive/json-ft-source/scripts/build_preference_pairs.py",
    "--config",
    "/content/drive/MyDrive/json-ft-source/configs/dpo.yaml",
    "--profile",
    "dev",
    "--run-name",
    RUN_NAME,
    "--runtime-root",
    RUNTIME_ROOT,
    "--input-path",
    SOURCE_DATASET,
]

print("Running command:")
print(" ".join(command))
result = subprocess.run(command, capture_output=True, text=True, check=False)
print("\nSTDOUT:\n")
print(result.stdout)
if result.stderr:
    print("\nSTDERR:\n")
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Preference pair build failed with exit code {result.returncode}")


Running command:
python /content/drive/MyDrive/json-ft-source/scripts/build_preference_pairs.py --config /content/drive/MyDrive/json-ft-source/configs/dpo.yaml --profile dev --run-name pref-dev-colab --runtime-root /content/drive/MyDrive/json-ft-runs --input-path /content/drive/MyDrive/json-ft-source/data/fixtures/support_tickets.jsonl

STDOUT:

Preference pair generation
Config: /content/drive/MyDrive/json-ft-source/configs/dpo.yaml
Profile: dev
Run name: pref-dev-colab
Input path: /content/drive/MyDrive/json-ft-source/data/fixtures/support_tickets.jsonl
Source split: train
Model name or path: Qwen/Qwen2.5-1.5B-Instruct
Adapter path: /content/drive/MyDrive/json-ft-runs/persistent/checkpoints/sft/sft-qwen2.5-1.5b-qlora-v1/adapter
Prompt source: messages
Source rows: 3
Candidate count per prompt: 3
is_colab=True
repo_root=/content/drive/MyDrive/json-ft-source
runtime_root=/content/drive/MyDrive/json-ft-runs
persistent_root=/content/drive/MyDrive/json-ft-runs/persistent
scratch_root=/con

In [2]:
from pathlib import Path
import json

run_dir = Path(RUNTIME_ROOT) / "persistent" / "preferences" / RUN_NAME
pairs_path = run_dir / f"{RUN_NAME}_dpo_pairs.jsonl"
audit_path = run_dir / f"{RUN_NAME}_preference_audit.jsonl"
summary_path = run_dir / f"{RUN_NAME}_preference_summary.json"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
audit_rows = [json.loads(line) for line in audit_path.read_text(encoding="utf-8").splitlines() if line.strip()]
pair_rows = [json.loads(line) for line in pairs_path.read_text(encoding="utf-8").splitlines() if line.strip()]

print(json.dumps(summary, indent=2, sort_keys=True))
print(f"Loaded {len(pair_rows)} DPO pairs and {len(audit_rows)} audit rows from {run_dir}")


{
  "adapter_path": "/content/drive/MyDrive/json-ft-runs/persistent/checkpoints/sft/sft-qwen2.5-1.5b-qlora-v1/adapter",
  "average_chosen_vs_rejected_score_gap": {
    "actions_f1_gap": 0.0,
    "hallucinated_key_reduction": 1.3333333333333333,
    "parses_json_gap": 0.0,
    "schema_valid_gap": 0.6666666666666666,
    "structured_field_match_gap": 1.6666666666666667,
    "summary_f1_gap": -0.035276945276945255,
    "summary_word_reduction": 1.6666666666666667
  },
  "candidate_count_total": 9,
  "candidate_json_valid_rate": 1.0,
  "candidate_schema_pass_rate": 0.6666666666666666,
  "chosen_schema_valid_rate": 1.0,
  "config_path": "/content/drive/MyDrive/json-ft-source/configs/dpo.yaml",
  "input_path": "/content/drive/MyDrive/json-ft-source/data/fixtures/support_tickets.jsonl",
  "model_name_or_path": "Qwen/Qwen2.5-1.5B-Instruct",
  "output_dir": "/content/drive/MyDrive/json-ft-runs/persistent/preferences/pref-dev-colab",
  "pair_count": 3,
  "pair_emission_rate": 1.0,
  "profile": "

In [3]:
skipped_rows = [row for row in audit_rows if row.get("skip_reason")]
emitted_rows = [row for row in audit_rows if not row.get("skip_reason")]

print("Skipped rows by reason:")
for reason, count in sorted(summary.get("skipped_counts", {}).items()):
    print(f"- {reason}: {count}")

if pair_rows:
    print("\nExample emitted DPO pair:")
    print(json.dumps(pair_rows[0], indent=2, sort_keys=True))

if emitted_rows:
    example = emitted_rows[0]
    print("\nExample emitted audit row")
    print(f"record_id: {example['record_id']}")
    print(f"decision_rationale: {example['decision_rationale']}")
    print(f"chosen_index: {example['chosen_index']}")
    print(f"rejected_index: {example['rejected_index']}")
    print("\nTop-ranked candidate scorecard:")
    print(json.dumps(example['candidates'][0]['scorecard'], indent=2, sort_keys=True))
    print("\nLowest-ranked candidate scorecard:")
    print(json.dumps(example['candidates'][-1]['scorecard'], indent=2, sort_keys=True))

if skipped_rows:
    skipped = skipped_rows[0]
    print("\nExample skipped row")
    print(f"record_id: {skipped['record_id']}")
    print(f"skip_reason: {skipped['skip_reason']}")
    print(f"decision_rationale: {skipped['decision_rationale']}")


Skipped rows by reason:

Example emitted DPO pair:
{
  "chosen": "{\n  \"actions_requested\": [\n    \"Refund the duplicate charge\",\n    \"Confirm that auto-pay is still active\"\n  ],\n  \"customer\": {\n    \"account_id\": \"AC-100\",\n    \"name\": \"Ava Cole\",\n    \"plan_tier\": \"pro\"\n  },\n  \"issue_category\": \"billing\",\n  \"priority\": \"urgent\",\n  \"product_area\": \"billing_portal\",\n  \"requires_human_followup\": true,\n  \"sentiment\": \"negative\",\n  \"summary\": \"Duplicate Invoice Charge\"\n}",
  "prompt": "You are a structured support-ticket extraction assistant.\nReturn only valid JSON that follows the schema exactly.\nUse null for unknown customer fields.\nUse [] when the customer did not request any explicit action.\nDo not add keys that are not defined by the schema.\n\nSchema name: support_ticket_extraction\nSchema version: 1.0.0\nTop-level JSON object fields:\n- summary: string\n- issue_category: billing | account_access | technical_bug | feature_requ

## Manual Spot-Check Checklist

- Confirm the emitted pair count is non-zero and the skip reasons are understandable.
- For at least 5 emitted rows, read the input text, gold payload, chosen candidate, and rejected candidate together.
- Verify the chosen response is schema-valid, has no hallucinated keys, and is more correct than rejected on the task labels.
- Verify the rejected response is genuinely worse, not just differently phrased.
- Check that summary differences reflect faithfulness and concision, not hidden semantic loss.
- Inspect at least 2 skipped rows and confirm the skip reason is appropriate before trusting the emitted-pair rate.
- Do not start DPO training until the audit sample looks clean enough to defend in `docs/05_failure_cases.md`.
